In [2]:
import pandas as pd
import numpy as np
import sys
import glob
from scipy.ndimage import median_filter, gaussian_filter
from scipy import signal
import math
import matplotlib.pyplot as plt
import mwdust
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.interpolate import CubicSpline
from astropy import constants as const
from astropy.io import ascii
# import astrodash
import pickle
import itertools
import csv
from collections import defaultdict
import json

In [3]:
sys.path.append("/Users/pnr5sh/Documents/phd/mmmp/")
import sidchaini.sidhelpers as sidhelpers

In [4]:
#reading in meta data from my dir
dataset = pd.read_csv('maven_data/bs_ZTFBTS_TransientTable.csv', header='infer')
dataset.columns

Index(['ZTFID', 'IAUID', 'RA', 'Dec', 'peakt', 'peakfilt', 'peakmag', 'type',
       'redshift', 'double-peaked', 'A_V', 'filenames'],
      dtype='object')

In [5]:
def csv_to_nested_dict(file_path):
    data = defaultdict(list)

    with open(file_path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            band = row["band"].strip().lower()

            entry = {
                "mjd": float(row["time"]),
                "mag": float(row["mag"]),
                "err": float(row["magerr"]),
            }

            data[band].append(entry)

    return dict(data)

In [6]:
# need to run above function for: SN 2020adnx, SN 2020urc, SN 2023gwl, SN 2024zsw

objs2convert = ['SN2020adnx', 'SN2020urc', 'SN2023gwl', 'SN2024zsw', 'SN2022hnt']

for i,obj in enumerate(objs2convert):
    ztfname = dataset.loc[dataset['IAUID']==obj,'ZTFID'].iloc[0]
    print(ztfname)
    
    #get the files names corresponding to that ztf id, there may be more than 1
    files = glob.glob(f'maven_data/lightcurves/{ztfname}_*.csv')
    
    #if only one then convert to nested dict
    if len(files)==1:
        file_path = files[0]
        data = csv_to_nested_dict(file_path)

    #save nested dict in json for gopreaux
    with open(f'temp/gopreaux/{ztfname}_ztf_fp.json', "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    #get metadata about SN from dataset
    info = {
        "ra":dataset.loc[dataset['IAUID']==obj,'RA'].iloc[0],
        "dec":dataset.loc[dataset['IAUID']==obj,'Dec'].iloc[0],
        "z":dataset.loc[dataset['IAUID']==obj,'redshift'].iloc[0],
    }
    
    #save info as separate dict
    with open(f'temp/gopreaux/{ztfname}_info.json', "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4)

ZTF20adadlqm
ZTF20acgiglu
ZTF23aaialcw
ZTF24abpdzvm
ZTF22aafrjnw
